# Kathmandu traffic vision - train vehicles

Fine-tunes YOLO11s (COCO-pretrained) to 4 classes for Kathmandu traffic: **motorcycle, car (incl. vans, jeeps, microbuses), bus, truck**. Data: Indian Driving Dataset (12,000-image sample; research use), Indian traffic-video frames (CC BY 4.0), and Nepali bike-and-car photos (weighted x3). Auto-rickshaws, bicycles, people and other classes are dropped. Validation is reported per source; none of these is Kathmandu junction footage, which is still needed for a real test.

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:

import os, random, yaml
from pathlib import Path
IN, OUT = Path("/kaggle/input/datasets"), Path("/kaggle/working/data")
VAL_LISTS = {}

def label_of(img):
    parts = list(img.parts)
    if "images" in parts:                                       # YOLO layout: .../images/... -> .../labels/...
        i = len(parts) - 1 - parts[::-1].index("images")
        p = Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")
        if p.exists():
            return p
    return img.with_suffix(".txt")                              # some sets keep labels next to images

def add(tag, split, images, names, mapping, repeat=1):
    """Link images into OUT/<split>, rewriting labels: keep only source classes in `mapping`
    (source name -> target id); boxes of other classes are dropped, the image stays."""
    d = OUT / split
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)
    n_box, kept = 0, []
    for img in images:
        rows, lab = [], label_of(img)
        if lab.exists():
            for ln in lab.read_text(errors="ignore").splitlines():
                parts = ln.split()
                if len(parts) != 5 or not parts[0].isdigit():      # skip polygons / malformed rows
                    continue
                k = int(parts[0])
                if k < len(names) and names[k] in mapping:
                    rows.append(" ".join([str(mapping[names[k]])] + parts[1:]))
        for r in range(repeat):
            stem = f"{tag}_{r}_{img.parent.name}_{img.stem}".replace(" ", "_")
            link = d / "images" / (stem + img.suffix.lower())
            if not link.exists():
                link.symlink_to(img)
            (d / "labels" / (stem + ".txt")).write_text("\n".join(rows))
            kept.append(str(link))
        n_box += len(rows)
    if split == "val":
        VAL_LISTS[tag] = kept
    print(f"{tag:<10} {split:<5} {len(images):>6} images x{repeat}  {n_box:>7} boxes kept")

def imgs(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_random(images, val_share=0.15, seed=0):
    images = images[:]
    random.Random(seed).shuffle(images)
    n = int(len(images) * val_share)
    return images[n:], images[:n]

def write_yaml(names):
    (OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "val/images",
                                                 "names": dict(enumerate(names))}))
    for tag, files in VAL_LISTS.items():          # per-source validation sets, to see where the model is weak
        (OUT / f"val_{tag}.txt").write_text("\n".join(files))
        (OUT / f"data_{tag}.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images",
                                                            "val": f"val_{tag}.txt", "names": dict(enumerate(names))}))


In [ ]:

NAMES = ["motorcycle", "car", "bus", "truck"]
M, C, B, T = range(4)

idd = IN / "redzapdos123/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset"
idd_names = yaml.safe_load((idd / "data.yaml").read_text())["names"]
idd_map = {"motorcycle": M, "car": C, "caravan": C, "bus": B, "truck": T, "trailer": T}
sample = lambda xs, n: random.Random(0).sample(xs, min(n, len(xs)))
add("idd", "train", sample(imgs(idd / "train/images"), 12000), idd_names, idd_map)
add("idd", "val", sample(imgs(idd / "val/images"), 1500), idd_names, idd_map)

dau = IN / "daudshah/vehicle-detection-dataset"
dau_names = yaml.safe_load((dau / "data.yaml").read_text())["names"]
dau_map = {"motorbike": M, "car": C, "van": C, "bus": B, "truck": T}
add("india_vid", "train", imgs(dau / "train/images"), dau_names, dau_map)
add("india_vid", "val", imgs(dau / "valid/images"), dau_names, dau_map)

nep = IN / "kshitizgajurel042/nepali-bike-and-car-images-with-annotations/Final Data"
nep_names = ["car", "license_plate", "bike"]
nep_map = {"bike": M, "car": C}
add("nepal", "train", imgs(nep / "images/train"), nep_names, nep_map, repeat=3)   # small but local: weighted x3
add("nepal", "val", imgs(nep / "images/val"), nep_names, nep_map)
write_yaml(NAMES)


In [ ]:

import glob, shutil
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data=str(OUT / "data.yaml"), epochs=60, time=5.0, imgsz=640, batch=32, patience=12,
            project="/kaggle/working/runs", name="vehicles", workers=4, seed=0, plots=True)
best = sorted(glob.glob("/kaggle/working/runs/vehicles*/weights/best.pt"))[-1]
shutil.copy(best, "/kaggle/working/vehicles_best.pt")
print("saved /kaggle/working/vehicles_best.pt")


In [ ]:

import pandas as pd
m = YOLO("/kaggle/working/vehicles_best.pt")
rows = []
for tag in VAL_LISTS:
    r = m.val(data=str(OUT / f"data_{tag}.yaml"), imgsz=640, verbose=False, plots=False)
    row = {"source": tag, "images": len(VAL_LISTS[tag]), "mAP50": round(r.box.map50, 3), "mAP50-95": round(r.box.map, 3)}
    for i, n in enumerate(NAMES):
        if i < len(r.box.ap50):
            row[f"AP50 {n}"] = round(float(r.box.ap50[i]), 3)
    rows.append(row)
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/vehicles_per_source.csv", index=False)
print(res.to_string(index=False))
shutil.rmtree(OUT)    # links only; keep the output small
